# Transformer 网络应用：问答

欢迎来到第四周的第三个实验，也是本课程的最后一个实验！恭喜你已经走到这一步了。在此 Notebook 中，你将探索你构建的 Transformer 架构的另一个应用程序。

**在此作业之后，你将能够**：

* 执行抽取式问答 
* 将预训练 Transformer 模型微调为自定义数据集
* 在 TensorFlow 和 PyTorch 中实现 QA 模型

## 目录


- [1 - 抽取式问答](#1)
    - [1.1 - 数据清理](#1-1)
    - [1.2 - 使用 🤗 库进行分词并对齐标签](#1-2)
- [2 - 训练](#2)
    - [2.1 TensorFlow 实现](#2-1)
    - [2.2 PyTorch 实现](#2-2)


<a name='1'></a>
## 1 - 抽取式问答

问答（QA）是一项自然语言处理任务，旨在自动回答问题。 *抽取式* QA 的目标是识别文本中包含问题答案的部分。例如，当任务是回答“简什么时候去非洲？”的问题时。给定文本数据“Jane Visits Africa in September”，问答模型将突出显示“September”。

* 你将使用在上一个作业中构建的 Transformer 模型的变体来回答有关故事的问题。
* 你将在 TensorFlow 和 PyTorch 中实现抽取式 QA 模型。

**推荐：**
* 如果你有兴趣，请查看我们的 [自然语言处理专项课程](https://www.coursera.org/specializations/natural-language-processing?=) 的 [课程 4：使用注意力模型的自然语言处理](https://www.coursera.org/learn/attention-models-in-nlp/home/welcome)，你可以在其中学习如何构建 Transformer 并使用 [Trax](https://trax.readthedocs.io/en/latest/) 库执行 QA。 

<a name='1-1'></a>
### 1.1 - 数据预处理

运行以下单元格来加载 [QA bAbI 数据集](https://research.fb.com/downloads/babi/)，这是 Facebook AI Research 生成的 bAbI 数据集之一，用于推进自然语言处理。

In [ ]:
from datasets import load_from_disk

# 加载数据集并打印训练集中的第一个示例
babi_dataset = load_from_disk('data/')
print(babi_dataset['train'][0])

看一下数据的格式。对于给定的故事，有两个句子作为上下文，以及一个问题。每个短语都有一个 ID。还有一个支持事实 ID，它指的是故事中有助于回答问题的句子。例如，对于问题“走廊东边是什么？”，支持事实“卧室在走廊东边”的 ID 为“2”。该问题还有答案“卧室”。

In [ ]:
babi_dataset['train'][102]

检查整个故事数据集是否具有这种格式。

In [ ]:
type_set = set()
for story in babi_dataset['train']:
    if str(story['story']['type'] )not in type_set:
        type_set.add(str(story['story']['type'] ))

In [ ]:
type_set

为了使数据更易于使用，你将展平数据集，将其从字典结构转换为表结构。

In [ ]:
flattened_babi = babi_dataset.flatten()

In [ ]:
flattened_babi

In [ ]:
next(iter(flattened_babi['train']))

现在访问你需要的信息变得更加容易！现在，你可以轻松地从故事中提取答案、问题和事实，还可以将事实加入到“句子”下的单个条目中。

In [ ]:
def get_question_and_facts(story):
    dic = {}
    dic['question'] = story['story.text'][2]
    dic['sentences'] = ' '.join([story['story.text'][0], story['story.text'][1]])
    dic['answer'] = story['story.answer'][2]
    return dic

In [ ]:
processed = flattened_babi.map(get_question_and_facts)

In [ ]:
processed['train'][2]

In [ ]:
processed['test'][2]

抽取式 QA 的目标是找到文本中包含问题答案的部分。你将使用字符串的索引来识别答案的位置。例如，如果某个问题的答案是“September”，则你需要在上下文句子“Jane Visits Africa in September”中查找单词“September”的开始和结束字符串索引。


使用下一个函数获取数据集中每个故事的答案的开始和结束索引。

In [ ]:
def get_start_end_idx(story):
    str_idx = story['sentences'].find(story['answer'])
    end_idx = str_idx + len(story['answer'])
    return {'str_idx':str_idx,
          'end_idx': end_idx}

In [ ]:
processed = processed.map(get_start_end_idx)

In [ ]:
num = 187
print(processed['test'][num])
start_idx = processed['test'][num]['str_idx']
end_idx = processed['test'][num]['end_idx']
print('answer:', processed['test'][num]['sentences'][start_idx:end_idx])

<a name='1-2'></a>
### 1.2 - 使用 🤗 库进行分词和对齐

现在，训练 Transformer 问答模型所需的数据已准备好。接下来进行分词和标签对齐，你可能已在命名实体识别实验中接触过这一流程。使用 [🤗 Transformer tokenizer](https://huggingface.co/transformers/main_classes/tokenizer.html) 对文本分词，并确保 tokenizer 与 Transformer 模型匹配。本练习使用 🤗 [DistilBERT fast tokenizer](https://huggingface.co/transformers/model_doc/distilbert.html)，将序列统一为 512 个 token，不足部分用 0 填充。

Transformer 模型通常由 tokenizer 进行训练，将单词拆分为子单词。例如，“非洲”一词可能会被拆分为多个子标记。这可能会在数据集的标签列表与 tokenizer 生成的标签列表之间产生一些不对齐，因为 tokenizer 可以将一个单词拆分为多个单词，或者添加特殊标记。在处理之前，务必将开始索引和结束索引与与 `tokenize_and_align()` 函数的目标答案词关联的标记对齐。在这种情况下，由于你对答案的开始和结束索引感兴趣，因此你需要对齐句子的索引以匹配单词标记的索引。 


In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('tokenizer/')

In [ ]:
def tokenize_align(example):
    encoding = tokenizer(example['sentences'], example['question'], truncation=True, padding=True, max_length=tokenizer.model_max_length)
    start_positions = encoding.char_to_token(example['str_idx'])
    end_positions = encoding.char_to_token(example['end_idx']-1)
    if start_positions is None:
        start_positions = tokenizer.model_max_length
    if end_positions is None:
        end_positions = tokenizer.model_max_length
    return {'input_ids': encoding['input_ids'],
          'attention_mask': encoding['attention_mask'],
          'start_positions': start_positions,
          'end_positions': end_positions}

In [ ]:
qa_dataset = processed.map(tokenize_align)

In [ ]:
qa_dataset = qa_dataset.remove_columns(['story.answer', 'story.id', 'story.supporting_ids', 'story.text', 'story.type'])

In [ ]:
qa_dataset['train'][200]

<font color='blue'><b>你应该记住：</b>
- *抽取式* QA 的目标是识别文本中包含问题答案的部分。
- Transformer 模型通常由 tokenizer 进行训练，将单词拆分为子单词。
  - 在处理之前，将开始索引和结束索引与目标答案词关联的标记对齐非常重要。
</font>

<a name='2'></a>
# 2 - 训练 

现在你已经完成了数据的分词和对齐，你可以将其输入预训练 🤗 Transformer 模型！你将使用 DistilBERT 模型，该模型与你用于预处理数据的 tokenizer 相匹配。

In [ ]:
train_ds = qa_dataset['train']
test_ds = qa_dataset['test']

In [ ]:
from transformers import TFDistilBertForQuestionAnswering
model = TFDistilBertForQuestionAnswering.from_pretrained("model/tensorflow", return_dict=False)

<a name='2-1'></a>
### 2.1 - TensorFlow 实现
对于此作业，你将执行两种实现，一种在 TensorFlow 中，另一种在 PyTorch 中。





#### 训练和测试数据集

**注：**
* 在 TensorFlow 实现中，你必须将数据格式类型设置为张量，这可能会创建不规则张量（不同长度的张量）。 
* 你必须使用 `to_tensor()` 方法将参差不齐的张量转换为正常张量，该方法会填充张量并将尺寸设置为 `[None, tokenizer.model_max_length]`，以便你可以根据批量大小将不同大小的张量输入模型中。  

In [ ]:
import tensorflow as tf

columns_to_return = ['input_ids','attention_mask', 'start_positions', 'end_positions']

train_ds.set_format(type='tf', columns=columns_to_return)

train_features = {x: train_ds[x] for x in ['input_ids', 'attention_mask']}
train_labels = {"start_positions": tf.reshape(train_ds['start_positions'], shape=[-1,1]),
                'end_positions': tf.reshape(train_ds['end_positions'], shape=[-1,1])}


train_tfdataset = tf.data.Dataset.from_tensor_slices((train_features, train_labels)).batch(8)

#### 训练 

终于到了开始训练模型的时候了！ 

* 使用 [tf.GradientTape()](https://www.tensorflow.org/api_docs/python/tf/GradientTape) 创建自定义训练函数
* 分别计算两个损失：一个对应答案的起始索引，另一个对应结束索引。
* `tf.GradientTape()` 记录前向传播过程中执行的操作，以便在反向传播过程中自动微分。 


In [ ]:
EPOCHS = 3
loss_fn1 = tf.keras.losses.SparseCategoricalCrossentropy( from_logits=True)
loss_fn2 = tf.keras.losses.SparseCategoricalCrossentropy( from_logits=True)
opt = tf.keras.optimizers.Adam(learning_rate=3e-5)

losses = []
for epoch in range(EPOCHS):
    print("Starting epoch: %d"% epoch )
    for step, (x_batch_train, y_batch_train) in enumerate(train_tfdataset):
        with tf.GradientTape() as tape:
            answer_start_scores, answer_end_scores = model(x_batch_train)
            loss_start = loss_fn1(y_batch_train['start_positions'], answer_start_scores)
            loss_end = loss_fn2(y_batch_train['end_positions'], answer_end_scores)
            loss = 0.5 * (loss_start + loss_end)
        losses.append(loss)
        grads = tape.gradient(loss, model.trainable_weights)
        opt.apply_gradients(zip(grads, model.trainable_weights))

        if step % 20 == 0:
            print("Training loss (for one batch) at step %d: %.4f"% (step, 
                                                                   float(loss_start)))


看看你的损失并尝试使用一些超参数以获得更好的结果！

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.show()

你已成功训练模型来帮助自动回答问题！尝试向它询问有关故事的问题。

In [ ]:
question, text = 'What is south of the bedroom?','The hallway is south of the garden. The garden is south of the bedroom.'
input_dict = tokenizer(text, question, return_tensors='tf')
outputs = model(input_dict)
start_logits = outputs[0]
end_logits = outputs[1]

all_tokens = tokenizer.convert_ids_to_tokens(input_dict["input_ids"].numpy()[0])
answer = ' '.join(all_tokens[tf.math.argmax(start_logits, 1)[0] : tf.math.argmax(end_logits, 1)[0]+1])
print(question, answer.capitalize())

恭喜！你刚刚在 TensorFlow 中实现了你的第一个 QA 模型。 

<a name='2-1'></a>
## 2.2 PyTorch 实现

[PyTorch](https://pytorch.org/) 是 Facebook 人工智能研究实验室开发的开源机器学习框架，可用于计算机视觉和自然语言处理。正如你可以想象的那样，它与 bAbI 数据集非常兼容。

#### 训练和测试数据集

继续尝试通过导入 PyTorch 创建训练和测试数据集。

In [ ]:
from torch.utils.data import DataLoader

columns_to_return = ['input_ids','attention_mask', 'start_positions', 'end_positions']
train_ds.set_format(type='pt', columns=columns_to_return)
test_ds.set_format(type='pt', columns=columns_to_return)

在 PyTorch 实现中，使用 [F1 分数](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) 作为评估指标，分别评估整个测试集上的答案起始与结束位置预测。

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(pred):
    start_labels = pred.label_ids[0]
    start_preds = pred.predictions[0].argmax(-1)
    end_labels = pred.label_ids[1]
    end_preds = pred.predictions[1].argmax(-1)
    
    f1_start = f1_score(start_labels, start_preds, average='macro')
    f1_end = f1_score(end_labels, end_preds, average='macro')
    
    return {
        'f1_start': f1_start,
        'f1_end': f1_end,
    }

#### 训练

现在加载一个预训练模型。 

**注意：** 你将使用 DistilBERT 而不是 TFDistilBERT 来实现 PyTorch。

In [ ]:
del model # 删除 TensorFlow 模型，以避免内存问题

In [ ]:
from transformers import DistilBertForQuestionAnswering

pytorch_model = DistilBertForQuestionAnswering.from_pretrained("model/pytorch")

你将使用 [🤗训练师](https://huggingface.co/transformers/main_classes/trainer.html)，而不是自定义训练循环，它包含一个基本训练循环，并且在 PyTorch 中相当容易实现。

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='results',          # 输出目录
    overwrite_output_dir=True,
    num_train_epochs=3,              # 训练周期总数
    per_device_train_batch_size=8,  # 训练期间每台设备的批量大小
    per_device_eval_batch_size=8,   # 用于评估的批量大小
    warmup_steps=20,                # 学习率调度器的预热步数
    weight_decay=0.01,               # 权重衰减强度
    logging_dir=None,            # 存放日志的目录
    logging_steps=50
)

trainer = Trainer(
    model=pytorch_model,                 # 已实例化、待训练的 🤗 Transformer 模型
    args=training_args,                  # 训练参数，定义如上
    train_dataset=train_ds,         # 训练数据集
    eval_dataset=test_ds,
    compute_metrics=compute_metrics             # 评估数据集
)

trainer.train()

In [ ]:
trainer.evaluate(test_ds)

现在可以向你的 PyTorch 模型提问了！ 
* 在用问题测试模型之前，你可以告诉 PyTorch 将模型和输入发送到 GPU（如果你的机器有 GPU），或者发送到 CPU（如果没有）。 
* 接下来，对输入进行分词，创建 PyTorch 张量，并将其发送到指定设备。 
* 管道的其余部分与你为 TensorFlow 实现的管道相对相似。   


In [ ]:
import torch

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

pytorch_model.to(device)

question, text = 'What is east of the hallway?','The kitchen is east of the hallway. The garden is south of the bedroom.'

input_dict = tokenizer(text, question, return_tensors='pt')

input_ids = input_dict['input_ids'].to(device)
attention_mask = input_dict['attention_mask'].to(device)

outputs = pytorch_model(input_ids, attention_mask=attention_mask)

start_logits = outputs[0]
end_logits = outputs[1]

all_tokens = tokenizer.convert_ids_to_tokens(input_dict["input_ids"].numpy()[0])
answer = ' '.join(all_tokens[torch.argmax(start_logits, 1)[0] : torch.argmax(end_logits, 1)[0]+1])

print(question, answer.capitalize())

### 恭喜！
 
你已完成本 Notebook，现在能够为问答任务实现 Transformer 模型！

你现在能够：
* 执行抽取式问答 
* 将预训练 Transformer 模型微调为自定义数据集
* 在 TensorFlow 和 PyTorch 中实现 QA 模型

<font color='blue'><b>你应该记住什么</b>：
- Transformer 模型通常配套使用将单词拆分为子词的 tokenizer。
  - 需要将答案的起始和结束位置与分词后的 token 对齐。
- PyTorch 相对轻量、易于使用，便于快速构建原型；TensorFlow 在扩展性方面具有优势，也广泛用于生产环境。
  - `tf.GradientTape` 允许你在 TensorFlow 中构建自定义训练循环
  - PyTorch 中的 `Trainer` API 提供了与 🤗 模型和数据集兼容的基本训练循环

In [ ]:
%%javascript
let element = document.getElementById('submit-notebook-button-group');
if (!element) {
    window._save_and_close = function(){
        IPython.notebook.save_checkpoint();
        IPython.notebook.session.delete();
        window.onbeforeunload = null
        setTimeout(function() {window.close();}, 1000)
    }
    let header = document.getElementById('maintoolbar-container');
    element = document.createElement("div");
    element.setAttribute("class", "btn-group");
    element.setAttribute("id", "submit-notebook-button-group");
    element.setAttribute("align", "right");
    element.setAttribute("style", "float:right")
    element.innerHTML = '<button class="btn btn-default" title="Save and close this notebook." style="background-color:rgb(42, 115, 204); color:white; padding:4px 8px" onclick=window._save_and_close()>Save and close</button>'
    header.appendChild(element); 
}                    